# Chapter 5: Multi-Source Robust Beamforming with Subspace Tracking

Companion code for *Robust Adaptive Beamforming and Its Applications* by Ayman Elnashar (Wiley, forthcoming).

This notebook reproduces every simulation figure of Chapter 5 (Figures 5.1–5.10), the result tables (Tables 5.4–5.12) and the numbers quoted in the text. The chapter develops **MPR-RAB**, a multi-primary robust adaptive beamformer for sparse multipath channels in which the desired signal arrives through several clusters. At every update it solves

$$\min_{\mathbf{w}}\ \mathbf{w}^H \widehat{\mathbf{R}}_{in}\mathbf{w}\quad \text{s.t.}\quad \mathrm{Re}\{\mathbf{w}^H\tilde{\mathbf{a}}_k\} \ge \nu_k\|\mathbf{w}\| + \beta_k,\quad k = 1,\dots,K,$$

with $\widehat{\mathbf{R}}_{in}$ the interference-plus-noise covariance reconstructed from the Capon spectrum (with off-grid refinement of strong peaks), margins $\nu_k = \kappa\sqrt{M/K}$ from joint feasibility, and gain targets $\beta_k$ and phases of $\tilde{\mathbf{a}}_k$ taken from the max-SINR beamformer of the estimated model. The problem is solved by the active-set KKT method of Chapter 2; the cluster directions are tracked on a deflated covariance with compensation of the delay of the covariance memory, and clusters are born and retired by a lifecycle test.

## Configuration

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))   # chapter folder, so that `ch5mprrab` imports without installation
QUICK = False             # True: 2 runs per cell (1 for training) instead of the chapter counts (smoke test)
REUSE_DATA = True         # True: reuse the simulation data in FIGDIR/data when it is complete (e.g. written by scripts/generate_figures.py)
LOCAL_TIMING = False      # True: time this machine for Figure 5.10 instead of using the timing recorded for the chapter
FIGDIR = os.path.abspath('../figures_quick' if QUICK else '../figures')
DATA = os.path.join(FIGDIR, 'data')
os.environ.update(FIGDIR=FIGDIR, CH5_DATA=DATA, CH5_INLINE='1', CH5_LOCAL_TIMING='1' if LOCAL_TIMING else '0')
%matplotlib inline
from ch5mprrab import scenarios as S
from ch5mprrab.figures import *
print('figures and data in', FIGDIR)

## Library

`ch5mprrab.channel` holds the signal model of Section 5.2: the multi-cluster channel with independent cluster gains, cluster drift, births and deaths, the expected SINR (5.4) and the oracle of Proposition 5.1. `ch5mprrab.algorithms` is the MPR-RAB kernel (`run_mpr_rab`): the margins of Section 5.3.2, the max-SINR targets and phases of Section 5.3.3 (`maxsinr_responses`), the active-set KKT solver of Section 5.3.5 (`as_snc`) with its residual certification, the reconstructed covariance with off-grid peak refinement of Section 5.4, and the tracking, delay compensation and lifecycle of Section 5.5. `ch5mprrab.planar` is the 8 × 8 planar array of Section 5.5.3 and its scenarios, `ch5mprrab.baselines` the comparators of Section 5.7.2 (LCMV-K, Par-K-RCB, LCMV-K-D, INCM-LCMV, INCM-MVDR, INCM-GEV), and `ch5mprrab.scenarios` the scenarios, protocol, metrics, selection rule and the stage runner.

## Settings (Section 5.7.1)

8 × 8 planar array (M = 64, main study) and 16-element linear array, half-wavelength spacing. SNR 5 dB per unit-power cluster, two interferers at INR 10 dB, at (+45°, 0°) and (−10°, −25°) on the planar array and at +45° and −60° on the linear array, noise power 1, forgetting factor 0.97. Nine scenarios A–I (Table 5.2) and six conditions (Table 5.3). Metrics: expected SINR (5.4) and minimum-cluster SINR (5.5), averaged over the last 50 snapshots of each run. The parameters of every family are selected on a training stream (4 runs per cell on the planar array, 8 on the linear array) and the results come from an independent stream with 30 runs per cell; MPR-RAB uses κ = 0.3 on the planar and κ = 0.4 on the linear array. Every run has a fixed seed and no method touches the global random stream, so every method of a cell sees identical data.

## Experiments

Each stage runs in its own process and writes its data and log into `FIGDIR/data` (about 5 hours on two cores for the full counts, about 30 minutes with `QUICK = True`). The planar training phase is run as well, since Figure 5.8 is drawn from it; the selection derived from it is compared with the selection used in the chapter.

In [ ]:
if REUSE_DATA and S.complete(DATA, quick=QUICK):
    print('reusing the simulation data in', DATA)
else:
    S.run_stages(DATA, jobs=2, quick=QUICK)
if LOCAL_TIMING:
    S.run_stages(DATA, stages=[S.TIMING], jobs=1, quick=QUICK)

## Figure 5.1: planar array, scenario A

Expected SINR per snapshot, mean of the 30 runs at the operating point (5 dB SNR, 10 dB INR, no pointing error). Two static clusters of equal power.

In [ ]:
fig_5_1()
show('Figure 5.1')

## Figure 5.2: planar array, scenario C

Both clusters drift at 0.025° per snapshot in azimuth and elevation. Top: expected SINR; bottom: mean angular error of the directions tracked by MPR-RAB.

In [ ]:
fig_5_2()
show('Figure 5.2')

## Figure 5.3: planar array, scenario D

A cluster is born at snapshot 150 (at −30° azimuth, 5° elevation) and the cluster at 0° dies at snapshot 280.

In [ ]:
fig_5_3()
show('Figure 5.3')

## Figure 5.4: expected against minimum-cluster SINR

Pooled over all 54 cells of the planar condition study. MPR-RAB for κ from 0 to 0.9 under the three target rules, the baselines at their trained parameters; the circle marks the selected setting (κ = 0.3, max-SINR targets) and the inset enlarges the region of the MPR-RAB curves and the reconstruction baselines.

In [ ]:
fig_5_4()
show('Figure 5.4')

## Figure 5.5: beam pattern, planar array, scenario H

Normalized gain of the final MPR-RAB weight of one run (seed 7) at the operating point; circles mark the clusters and crosses the interferers.

In [ ]:
fig_5_5()
show('Figure 5.5')

## Figure 5.6: planar array, scenario I (shadowing)

The strongest cluster is shadowed on snapshots 150 to 249 (grey interval).

In [ ]:
fig_5_6()
show('Figure 5.6')

## Figure 5.7: linear array, scenario C

Mean of 15 runs at the operating point on a stream separate from the condition study. Bottom: enlarged top of the traces with two reduced versions of MPR-RAB (no delay compensation; directions refined every 30 snapshots on the scan grid without delay compensation).

In [ ]:
fig_5_7()
show('Figure 5.7')

## Figure 5.8: sensitivity to κ, planar training study

Pooled expected (left) and minimum-cluster (right) SINR of the training stream against κ for the three target rules, with the levels of the reconstruction baselines; the vertical line marks the selected κ.

In [ ]:
fig_5_8()
show('Figure 5.8')

## Figure 5.9: interferer crossing a tracked cluster, linear array

Clusters at −20° and +20°, one interferer at −60° and a second one sweeping from +40° to 0° over 300 snapshots, crossing the second cluster at snapshot 150; mean of 10 runs, with and without the deflation of Section 5.5.1.

In [ ]:
fig_5_9()
show('Figure 5.9')

## Figure 5.10: time per snapshot against array size

Linear arrays, one core, median of 5 repetitions. Drawn from the timing recorded for the chapter (Table 5.1) unless `LOCAL_TIMING = True`; the timers differ in scope (Section 5.6.2), so the ratios are not end-to-end comparisons.

In [ ]:
fig_5_10()
show('Figure 5.10')

## Tables and quoted numbers

Tables 5.4–5.12, the ablations, the stress tests, the proposition and feasibility checks and the solver audit, collected into `figures/results_summary.json`.

In [ ]:
path = write_summary()
print('written', path)
for key in ['Table 5.4', 'Table 5.5', 'Table 5.6', 'Table 5.7', 'Table 5.8', 'Table 5.9', 'Table 5.10', 'Table 5.11', 'Table 5.12']:
    show(key)

In [ ]:
for key in ['Deflation at 35 dB INR', 'Tracking off', 'Interferers 0.25 deg off', 'Solver certification', 'Checks']:
    show(key)